

**Install Libraries**


In [1]:

!pip install datasets
!pip install faiss-cpu
!pip install \
transformers==4.46.3 \
huggingface_hub==0.26.2 \
tokenizers==0.20.3 \
sentence-transformers==3.3.1
!pip install keybert==0.8.5
!pip install langchain langchain-community langchain-core
!pip install -U langchain langchain-community langchain-huggingface
!pip install -U langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled tr

In [2]:
from datasets import load_dataset

**Load Dataset**

In [3]:
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

In [4]:
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [5]:
dataset[0]

{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

**dataset into a Pandas DataFrame.**

In [6]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [7]:
df.shape

(117592, 4)

In [8]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [9]:
df = df[['title', 'abstract']]

In [10]:
df.head()

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [11]:
df = df.head(15000)

In [12]:
df.shape

(15000, 2)

In [13]:
df.isnull().sum()

,0
title,0
abstract,0


In [14]:
df["paper_text"] = df["title"] + " " + df["abstract"]

In [15]:
df[["paper_text"]].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [16]:
type(df[["paper_text"]])

pandas.core.frame.DataFrame

In [17]:
df["paper_text"].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [18]:
type(df["paper_text"])

pandas.core.series.Series

In [19]:
print(df["paper_text"].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

In [20]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

In [21]:
print(df["paper_text"].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regress

**Sentence Transformers**

In [22]:
from sentence_transformers import SentenceTransformer

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [23]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
print(type(model))

<class 'sentence_transformers.SentenceTransformer.SentenceTransformer'>


In [25]:
sample_text = df["paper_text"].iloc[0]

embedding = model.encode(sample_text)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [26]:
embedding[:10]

array([-0.13156408, -0.00678268, -0.00367609,  0.03265162,  0.11219638,
        0.01227266,  0.09816723, -0.09005228,  0.04231165, -0.0197735 ],
      dtype=float32)

In [27]:
sample_embeddings = model.encode(
    df["paper_text"].head(5).tolist()
)

print(type(sample_embeddings))
print(sample_embeddings.shape)

<class 'numpy.ndarray'>
(5, 384)


In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
similarity = cosine_similarity(
    sample_embeddings[0].reshape(1, -1),
    sample_embeddings[1].reshape(1, -1)
)

print(similarity)

[[0.36625272]]


In [30]:
similarity = cosine_similarity(
    sample_embeddings[0].reshape(1,-1),
    sample_embeddings[0].reshape(1,-1)
)

print(similarity)

[[1.]]


In [31]:
for i in range(1,5):

    sim = cosine_similarity(
        sample_embeddings[0].reshape(1,-1),
        sample_embeddings[i].reshape(1,-1)
    )

    print(f"Paper 1 vs Paper {i+1}: {sim[0][0]:.4f}")

Paper 1 vs Paper 2: 0.3663
Paper 1 vs Paper 3: 0.3352
Paper 1 vs Paper 4: 0.1551
Paper 1 vs Paper 5: 0.3742


**Generate Full Embeddings**

In [32]:
import os
import numpy as np

if os.path.exists("paper_embeddings.npy"):

    print("Loading saved embeddings...")

    embeddings = np.load("paper_embeddings.npy")

else:

    print("Generating embeddings...")

    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    np.save("paper_embeddings.npy", embeddings)

    print("Embeddings saved successfully!")#embeddings = model.encode(
    #df["paper_text"].tolist(),
    #batch_size=32,
    #show_progress_bar=True
#)

Generating embeddings...


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Embeddings saved successfully!


In [33]:
print(type(embeddings))
print(embeddings.shape)


<class 'numpy.ndarray'>
(15000, 384)


In [34]:
embeddings.nbytes / (1024*1024)

21.97265625

In [35]:
embeddings.dtype


dtype('float32')

In [36]:
import os
import faiss

if os.path.exists("paper_faiss.index"):

    print("Loading existing FAISS index...")

    index = faiss.read_index("paper_faiss.index")

else:

    print("Creating new FAISS index...")

    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(384)

    index.add(embeddings)

    faiss.write_index(index, "paper_faiss.index")

    print("FAISS index saved successfully!")#faiss.normalize_L2(embeddings)

#index = faiss.IndexFlatIP(384)

#index.add(embeddings)

Creating new FAISS index...
FAISS index saved successfully!


In [37]:
import numpy as np

np.linalg.norm(embeddings[0])

np.float32(1.0)

In [38]:
print(index.ntotal)

15000


In [39]:
query = "deep learning for medical image analysis"
query_embedding = model.encode([query])
query_embedding.shape


(1, 384)

In [40]:
faiss.normalize_L2(query_embedding)


np.linalg.norm(query_embedding[0])

np.float32(1.0)

In [41]:
D, I = index.search(query_embedding, 5)

print(D)
print(I)

[[0.6807246  0.67092216 0.6521999  0.62811756 0.6131154 ]]
[[10466 13730 11873 12691 11282]]


In [42]:
print(df.iloc[10466]["title"])

A Perspective on Deep Imaging


In [43]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [44]:
def search_papers(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    return D, I

In [45]:
D, I = search_papers(
    "deep learning for medical image analysis"
)

print(D)
print(I)

[[0.6807246  0.67092216 0.6521999  0.62811756 0.6131154 ]]
[[10466 13730 11873 12691 11282]]


In [46]:
def search_papers(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        print("Abstract:")

        print(df.iloc[idx]["abstract"][:500])

        print()

In [47]:
search_papers(
    "deep learning for medical image analysis"
)

Similarity: 0.6807
Title: A Perspective on Deep Imaging

Abstract:
  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity: 0.6709
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?

Abstract:
  Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a large set 

In [48]:
import transformers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Transformers Path:", transformers.__file__)

print("Hub:", huggingface_hub.__version__)
print("Hub Path:", huggingface_hub.__file__)

import transformers.utils as u

print(hasattr(u, "HUGGINGFACE_CO_RESOLVE_ENDPOINT"))

Transformers: 4.46.3
Transformers Path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Hub: 0.36.2
Hub Path: /usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py
True


In [49]:
import transformers.utils as u

print(hasattr(u, "HUGGINGFACE_CO_RESOLVE_ENDPOINT"))

True


In [50]:
from transformers import pipeline

In [51]:
type(pipeline)

function

In [52]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [53]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [54]:
summary = summarizer(
    df.iloc[10466]["abstract"],
    max_length=120,
    min_length=40,
    do_sample=False
)

In [55]:
print(summary)

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]


In [56]:
type(summary)

list

In [57]:
summary[0]

{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}

In [58]:
type(summary[0])

dict

In [59]:
summary[0]["summary_text"]

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

In [60]:
print(summary[0]["summary_text"])

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.


In [61]:
for score, idx in zip(D[0], I[0]):

    print("=" * 100)

    print("Similarity:", round(float(score), 4))

    print("Title:", df.iloc[idx]["title"])

    print()

    summary = summarizer(
        df.iloc[idx]["abstract"],
        max_length=120,
        min_length=40,
        do_sample=False
    )

    print("Summary:")

    print(summary[0]["summary_text"])

    print()

Similarity: 0.6807
Title: A Perspective on Deep Imaging

Summary:
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity: 0.6709
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?

Summary:
Training a deep convolutional neural network from scratch is difficult because it requires a large amount of labeled training data. A promising alternative is to fine-tune a CNN that has been pre-trained using, for instance, a large set of labeled natural images.

Similarity: 0.6522
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection

Summary:
The classification of MRI images according to the anatomical field of vie

In [62]:
def summarize_text(text):

    summary = summarizer(
        text,
        max_length=120,
        min_length=40,
        do_sample=False
    )

    return summary[0]["summary_text"]

In [63]:
def search_and_summarize(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )

        print("Summary:")

        print(summary[0]["summary_text"])

        print()

In [64]:
search_and_summarize(
    "Deep Learning in Medical Imaging",
    k=5
)

Similarity: 0.7355
Title: A Perspective on Deep Imaging

Summary:
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity: 0.6882
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection

Summary:
The classification of MRI images according to the anatomical field of view is a necessary task to solve when faced with the increasing quantity of medical images. Using a common architecture (such as AlexNet) provides quite good results, but not sufficient for clinical use.

Similarity: 0.6532
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?

Summary:
Training a deep convolutional neural network from scratch is d

In [65]:
from keybert import KeyBERT

In [66]:
kw_model = KeyBERT(model)

In [67]:
type(kw_model)

keybert._model.KeyBERT

In [68]:
text = df.iloc[10466]["abstract"]

keywords = kw_model.extract_keywords(text)

In [69]:
print(type(keywords))

<class 'list'>


In [70]:
print(type(keywords[0]))

<class 'tuple'>


In [71]:
print(keywords[0])

('imaging', 0.4528)


In [72]:
keywords = kw_model.extract_keywords(
    text,
    keyphrase_ngram_range=(1,3),
    stop_words="english",
    top_n=5
)

print(keywords)

[('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]


In [73]:
def search_and_summarize(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )

        print("Summary:")

        print(summary[0]["summary_text"])

        print()

        keywords = kw_model.extract_keywords(
            df.iloc[idx]["abstract"],
            keyphrase_ngram_range=(1,3),
            stop_words="english",
            top_n=5
        )

        print("Keywords:")

        for keyword, score in keywords:
            print("-", keyword)

        print()

In [74]:
search_and_summarize(
    "Deep Learning in Medical Imaging",
    k=5
)

Similarity: 0.7355
Title: A Perspective on Deep Imaging

Summary:
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Keywords:
- tomographic imaging deep
- imaging deep learning
- learning medical imaging
- imaging deep
- medical imaging

Similarity: 0.6882
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection

Summary:
The classification of MRI images according to the anatomical field of view is a necessary task to solve when faced with the increasing quantity of medical images. Using a common architecture (such as AlexNet) provides quite good results, but not sufficient for clinical use.

Keywords:
- classification mri images
- classification mri
- mri d

**Hybrid NER Architecture**

In [75]:
ner = pipeline(
    "ner",
    aggregation_strategy="simple"
)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [76]:
text = """
ResNet50 was trained on ImageNet using PyTorch
at Stanford University.
"""
entities = ner(text)

print(entities)

[{'entity_group': 'ORG', 'score': np.float32(0.89313823), 'word': 'ResNet50', 'start': 1, 'end': 9}, {'entity_group': 'ORG', 'score': np.float32(0.8266236), 'word': 'ImageNet', 'start': 25, 'end': 33}, {'entity_group': 'ORG', 'score': np.float32(0.94511735), 'word': 'PyTorch', 'start': 40, 'end': 47}, {'entity_group': 'ORG', 'score': np.float32(0.9337442), 'word': 'Stanford University', 'start': 51, 'end': 70}]


In [77]:

from langchain_core.tools import tool
@tool
def search_papers(query: str) -> str:
    """
    Search research papers from the FAISS database
    and return the most relevant results.
    """

In [141]:
from langchain_huggingface import HuggingFacePipeline

In [140]:
llm = HuggingFacePipeline(
    pipeline=summarizer
)

In [142]:
llm

HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, pipeline=<transformers.pipelines.text2text_generation.SummarizationPipeline object at 0x7b7064db9d00>, model_id='facebook/bart-large-cnn')

In [143]:
response = llm.invoke(
    "Deep Learning has become one of the most successful techniques for medical image reconstruction using convolutional neural networks."
)

Your max_length is set to 142, but your input_length is only 23. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=11)


In [82]:
response

"Deep Learning has become one of the most successful techniques for medical image reconstruction using convolutional neural networks. Deep Learning is a technique that uses deep neural networks to learn about a patient's medical history. It can also be used to learn more about a person's physical condition."

In [83]:
print(type(response))

<class 'str'>


In [84]:
from langchain_groq import ChatGroq

In [85]:
from google.colab import userdata

In [86]:
api_key = userdata.get("GROQ_API_KEY")

In [87]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key,
    temperature=0
)

In [126]:
response = llm.invoke("Hello, who are you?")

In [89]:
response

AIMessage(content="I'm an artificial intelligence model known as a large language model (LLM) or conversational AI. I'm a computer program designed to understand and respond to human language in a way that's natural and helpful.\n\nI don't have a personal identity or emotions like humans do, but I'm here to assist you with any questions, topics, or tasks you'd like to discuss. I can provide information, answer questions, generate text, and even engage in conversation.\n\nI'm constantly learning and improving my responses based on the data and interactions I receive, so feel free to ask me anything and I'll do my best to help!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 41, 'total_tokens': 169, 'completion_time': 0.154060949, 'completion_tokens_details': None, 'prompt_time': 0.002769954, 'prompt_tokens_details': None, 'queue_time': 0.049256402, 'total_time': 0.156830903}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint'

In [90]:
response.content

"I'm an artificial intelligence model known as a large language model (LLM) or conversational AI. I'm a computer program designed to understand and respond to human language in a way that's natural and helpful.\n\nI don't have a personal identity or emotions like humans do, but I'm here to assist you with any questions, topics, or tasks you'd like to discuss. I can provide information, answer questions, generate text, and even engage in conversation.\n\nI'm constantly learning and improving my responses based on the data and interactions I receive, so feel free to ask me anything and I'll do my best to help!"

In [91]:
from langchain_core.tools import tool

In [92]:


@tool
def search_and_summarize(query: str, k: int = 5) -> str:
    """
    Search research papers from the FAISS database,
    retrieve the top-k most similar papers,
    summarize each paper using BART,
    and return the results.
    """


    query_embedding = model.encode([query])


    faiss.normalize_L2(query_embedding)


    D, I = index.search(query_embedding, k)


    result = ""

    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

        paper = df.iloc[idx]


        summary = summarizer(
            paper["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]


        result += f"Rank: {rank}\n"
        result += f"Similarity Score: {round(float(score),4)}\n"
        result += f"Title: {paper['title']}\n\n"


        result += paper["abstract"] + "\n\n"


        result += summary + "\n\n"

    return result

In [93]:
tools = [search_and_summarize]

In [94]:
from langchain.agents import create_agent

In [95]:
agent = create_agent(
    model=llm,
    tools=tools
)

In [96]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find the top 3 research papers on Vision Transformer."
            }
        ]
    }
)

In [97]:
response

{'messages': [HumanMessage(content='Find the top 3 research papers on Vision Transformer.', additional_kwargs={}, response_metadata={}, id='37e61e0c-2407-4c84-9df7-14a64b176aaa'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qj442tbma', 'function': {'arguments': '{"k":3,"query":"Vision Transformer"}', 'name': 'search_and_summarize'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 290, 'total_tokens': 313, 'completion_time': 0.020313511, 'completion_tokens_details': None, 'prompt_time': 0.016616887, 'prompt_tokens_details': None, 'queue_time': 0.01824939, 'total_time': 0.036930398}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f790c-de19-7a73-997c-70f94057b2fd-0', tool_calls=[{'name': 'search_and_summarize', 'args': {'k': 3, 'query': 'Vision Transformer'}, 'id': 'q

In [98]:
print(type(response))

<class 'dict'>


In [99]:
print(response.keys())

dict_keys(['messages'])


In [100]:
len(response["messages"])

4

In [101]:
response["messages"][0]

HumanMessage(content='Find the top 3 research papers on Vision Transformer.', additional_kwargs={}, response_metadata={}, id='37e61e0c-2407-4c84-9df7-14a64b176aaa')

In [102]:
response["messages"][1]

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qj442tbma', 'function': {'arguments': '{"k":3,"query":"Vision Transformer"}', 'name': 'search_and_summarize'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 290, 'total_tokens': 313, 'completion_time': 0.020313511, 'completion_tokens_details': None, 'prompt_time': 0.016616887, 'prompt_tokens_details': None, 'queue_time': 0.01824939, 'total_time': 0.036930398}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f790c-de19-7a73-997c-70f94057b2fd-0', tool_calls=[{'name': 'search_and_summarize', 'args': {'k': 3, 'query': 'Vision Transformer'}, 'id': 'qj442tbma', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 23, 'total_tokens': 313})

In [103]:
response["messages"][2]

ToolMessage(content='Rank: 1\nSimilarity Score: 0.4415\nTitle: Dense Transformer Networks\n\n  The key idea of current deep learning methods for dense prediction is to\napply a model on a regular patch centered on each pixel to make pixel-wise\npredictions. These methods are limited in the sense that the patches are\ndetermined by network architecture instead of learned from data. In this work,\nwe propose the dense transformer networks, which can learn the shapes and sizes\nof patches from data. The dense transformer networks employ an encoder-decoder\narchitecture, and a pair of dense transformer modules are inserted into each of\nthe encoder and decoder paths. The novelty of this work is that we provide\ntechnical solutions for learning the shapes and sizes of patches from data and\nefficiently restoring the spatial correspondence required for dense prediction.\nThe proposed dense transformer modules are differentiable, thus the entire\nnetwork can be trained. We apply the proposed 

In [104]:
response["messages"][3]

AIMessage(content='The top 3 research papers on Vision Transformer are:\n\n1. Dense Transformer Networks\n2. Inverse Compositional Spatial Transformer Networks\n3. gvnn: Neural Network Library for Geometric Computer Vision\n\nThese papers discuss the application of transformer networks in computer vision, including dense prediction, spatial transformation, and geometric computer vision. They propose new layers and architectures that can be used to learn invariance to 3D geometric transformation, perform end-to-end learning of a network involving domain knowledge, and enable backpropagation for parametric transformations.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 1106, 'total_tokens': 1217, 'completion_time': 0.149595312, 'completion_tokens_details': None, 'prompt_time': 0.061419463, 'prompt_tokens_details': None, 'queue_time': 0.019138494, 'total_time': 0.211014775}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 

In [105]:


@tool
def search_and_summarize(query: str, k: int = 5):
    """
    Search research papers from the FAISS database,
    retrieve the top-k most similar papers,
    summarize each paper using BART,
    and return the results as structured data.
    """

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)


    D, I = index.search(query_embedding, k)

    papers = []

    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

        paper = df.iloc[idx]

        summary = summarizer(
            paper["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]


        paper_data = {
            "rank": rank,
            "similarity": round(float(score), 4),
            "title": paper["title"],
            "abstract": paper["abstract"],
            "summary": summary
        }

        papers.append(paper_data)

    return papers

In [106]:


@tool
def extract_keywords(text: str, top_n: int = 5) -> str:
    """
    Extract the most important keywords from the given text
    using the KeyBERT model.
    """


    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_n
    )

    result = "Top Keywords:\n\n"

    for rank, (keyword, score) in enumerate(keywords, start=1):

        result += (
            f"{rank}. {keyword} "
            f"(Relevance Score: {round(score, 4)})\n"
        )

    return result

In [107]:
tools = [
    search_and_summarize,
    extract_keywords
]

In [108]:
agent = create_agent(
    model=llm,
    tools=tools
)

In [109]:
print(tools)

[StructuredTool(name='search_and_summarize', description='Search research papers from the FAISS database,\nretrieve the top-k most similar papers,\nsummarize each paper using BART,\nand return the results as structured data.', args_schema=<class 'langchain_core.utils.pydantic.search_and_summarize'>, func=<function search_and_summarize at 0x7b70189ff380>), StructuredTool(name='extract_keywords', description='Extract the most important keywords from the given text\nusing the KeyBERT model.', args_schema=<class 'langchain_core.utils.pydantic.extract_keywords'>, func=<function extract_keywords at 0x7b70189ff2e0>)]


In [111]:
papers = search_and_summarize.invoke({"query": "Deep Learning for Medical Image Reconstruction"})

abstract = papers[0]["abstract"]

keywords = extract_keywords.invoke({"text": abstract})

print(keywords)

Top Keywords:

1. imaging deep (Relevance Score: 0.5919)
2. medical imaging (Relevance Score: 0.5281)
3. tomographic imaging (Relevance Score: 0.525)
4. image reconstruction (Relevance Score: 0.5095)
5. imaging (Relevance Score: 0.4528)



In [128]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "First search the research paper titled Deep Learning for Medical Image Reconstruction Then use the abstract of the most relevant paper to extract the top 5 keywords."
            }
        ]
    }
)

In [129]:
response = extract_keywords.invoke(
    {
        "text": "Deep Learning for Medical Image Reconstruction",
        "top_n": 5
    }
)

print(response)

Top Keywords:

1. image reconstruction (Relevance Score: 0.5655)
2. deep learning (Relevance Score: 0.526)
3. medical image (Relevance Score: 0.4839)
4. learning medical (Relevance Score: 0.4024)
5. reconstruction (Relevance Score: 0.3745)



In [130]:

user_query = "Extract the top 5 keywords from Deep Learning for Medical Image Reconstruction."

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(user_query)

print(response)



print(response.tool_calls)




tool_call = response.tool_calls[0]

print(tool_call)

content='' additional_kwargs={'tool_calls': [{'id': 'qzcv2xvby', 'function': {'arguments': '{"text":"Deep Learning for Medical Image Reconstruction","top_n":5}', 'name': 'extract_keywords'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 379, 'total_tokens': 405, 'completion_time': 0.029602589, 'completion_tokens_details': None, 'prompt_time': 0.022288563, 'prompt_tokens_details': None, 'queue_time': 0.049757914, 'total_time': 0.051891152}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4848f70c04', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f792f-f82c-7261-a389-3c65c678b9aa-0' tool_calls=[{'name': 'extract_keywords', 'args': {'text': 'Deep Learning for Medical Image Reconstruction', 'top_n': 5}, 'id': 'qzcv2xvby', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 379, 'output_tokens': 26, 'total_tokens': 405}
[{'name

In [131]:
tool_name = tool_call["name"]

print(tool_name)

extract_keywords


In [132]:
tool_args = tool_call["args"]

print(tool_args)

{'text': 'Deep Learning for Medical Image Reconstruction', 'top_n': 5}


In [133]:
if tool_name == "extract_keywords":

    tool_result = extract_keywords.invoke(tool_args)

elif tool_name == "search_and_summarize":

    tool_result = search_and_summarize.invoke(tool_args)

print(tool_result)

Top Keywords:

1. image reconstruction (Relevance Score: 0.5655)
2. deep learning (Relevance Score: 0.526)
3. medical image (Relevance Score: 0.4839)
4. learning medical (Relevance Score: 0.4024)
5. reconstruction (Relevance Score: 0.3745)



In [134]:
from langchain_core.messages import ToolMessage

In [135]:
tool_message = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call["id"]
)

In [136]:
print(tool_message)

content='Top Keywords:\n\n1. image reconstruction (Relevance Score: 0.5655)\n2. deep learning (Relevance Score: 0.526)\n3. medical image (Relevance Score: 0.4839)\n4. learning medical (Relevance Score: 0.4024)\n5. reconstruction (Relevance Score: 0.3745)\n' tool_call_id='qzcv2xvby'


In [137]:
from langchain_core.messages import SystemMessage, HumanMessage

final_response = llm.invoke(
    [
        SystemMessage(
            content="""
You are a helpful AI assistant.

Rules:
1. Always use the tool output.
2. Never ignore tool results.
3. Present the complete tool output.
4. Add a short explanation after the tool output if necessary.
"""
        ),

        HumanMessage(content=user_query),

        response,

        tool_message
    ]
)

print(final_response.content)

The top 5 keywords extracted from the given text are related to the field of medical image reconstruction using deep learning techniques. These keywords can be useful for search engine optimization (SEO), topic modeling, or information retrieval tasks.


In [138]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

final_response = llm.invoke(
[
SystemMessage(
content="""
You are a research assistant.

IMPORTANT RULES

1. Never ignore tool output.

2. Always display every keyword returned by the tool.

3. Do not summarize the keywords.

4. Print the keywords exactly as received.

5. After printing them, give a short explanation.
"""
),

HumanMessage(content=user_query),

response,

tool_message

]
)

print(final_response.content)

Explanation: The top 5 keywords extracted from the given topic are related to the application of deep learning techniques in medical image reconstruction. These keywords highlight the importance of using deep learning algorithms to improve the quality and accuracy of medical images, which is a crucial aspect of medical imaging.


In [139]:
from langchain_core.tools import tool

@tool
def compare_papers(
    paper1: str,
    paper2: str
) -> str:
    """
    Compare two research papers based on their abstracts.
    The tool retrieves the closest matching papers from the FAISS database
    and uses the LLM to compare them.
    """


    embedding1 = model.encode([paper1])

    faiss.normalize_L2(embedding1)

    D1, I1 = index.search(embedding1, 1)

    first_paper = df.iloc[I1[0][0]]


    embedding2 = model.encode([paper2])

    faiss.normalize_L2(embedding2)

    D2, I2 = index.search(embedding2, 1)

    second_paper = df.iloc[I2[0][0]]


    comparison_prompt = f"""
Compare the following two research papers.

Paper 1

Title:
{first_paper['title']}

Abstract:
{first_paper['abstract']}


Paper 2

Title:
{second_paper['title']}

Abstract:
{second_paper['abstract']}


Compare them based on:

1. Research Objective

2. Methodology

3. Key Contributions

4. Advantages

5. Limitations

6. Applications

Present the comparison in a clear table.
"""


    response = llm.invoke(comparison_prompt)

    return response.content